# Personalized Health Alerts for Preventive Care Using ECG Data
### A Comparative Study of Random Forest and LSTM

**Authors:** Najma Nour, Haythem Elmessiry. Department of Artificial Intelligence, Canadian University Dubai.

This notebook takes single ECG heartbeats and classifies each one as **Normal** or **Abnormal**. It compares two models:
- **Random Forest** on 13 hand-crafted statistical features
- **LSTM** on the raw 187-point signal

The training set is balanced with SMOTE. The notebook ends with two ablation studies: the effect of SMOTE and robustness to noise.

## 1. Setup
Install the dependencies first: `pip install -r requirements.txt`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.model_selection import cross_val_score
from scipy import stats
import time

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print("✅ All libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")

## 2. Load the data
The [ECG Heartbeat Categorization Dataset](https://www.kaggle.com/datasets/shayanfazeli/heartbeat) is a preprocessed version of the MIT-BIH Arrhythmia Database. Each row is one heartbeat: 187 amplitude samples followed by a class label.

In [ ]:
from pathlib import Path

# Download mitbih_train.csv and mitbih_test.csv into data/ (see data/README.md).
# On Google Colab, mount Drive and point DATA_DIR at the folder holding the CSVs.
DATA_DIR = Path('../data')
FIG_DIR = Path('../results/figures')
MODEL_DIR = Path('../results/models')
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / 'mitbih_train.csv', header=None)
test_df  = pd.read_csv(DATA_DIR / 'mitbih_test.csv',  header=None)

print("✅ Dataset loaded successfully")
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTrain sample:\n{train_df.head(3)}")

## 3. Explore the raw labels

In [ ]:
print("=" * 50)
print("RAW LABEL DISTRIBUTION (Before Binary Conversion)")
print("=" * 50)

label_map = {
    0: 'Normal (N)',
    1: 'Supraventricular (S)',
    2: 'Ventricular (V)',
    3: 'Fusion (F)',
    4: 'Unknown (Q)'
}

raw_labels = train_df.iloc[:, -1].value_counts().sort_index()
for label, count in raw_labels.items():
    pct = count / len(train_df) * 100
    print(f"  Class {int(label)} - {label_map[int(label)]}: "
          f"{count:,} samples ({pct:.1f}%)")

# Plot raw distribution
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6', '#3498db']
ax.bar(
    [label_map[i] for i in range(5)],
    [raw_labels.get(i, 0) for i in range(5)],
    color=colors, edgecolor='black'
)
ax.set_title('Raw Label Distribution (5 Classes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Samples')
ax.set_xlabel('Class')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIG_DIR / 'raw_distribution.png', dpi=150)
plt.show()

## 4. Binary target: Normal vs Abnormal
Class 0 is Normal; classes 1 to 4 are merged into Abnormal.

In [ ]:
print("=" * 50)
print("BINARY CONVERSION")
print("  0 = Normal")
print("  1 = Abnormal (classes 1, 2, 3, 4 combined)")
print("=" * 50)

# Features (187 ECG points per heartbeat)
X_train_raw = train_df.iloc[:, :-1].values
X_test_raw  = test_df.iloc[:, :-1].values

# Raw labels
y_train_raw = train_df.iloc[:, -1].values
y_test_raw  = test_df.iloc[:, -1].values

# Binary conversion (key step)
y_train = (y_train_raw > 0).astype(int)
y_test  = (y_test_raw  > 0).astype(int)

# Report binary distribution
print(f"\nTraining Set:")
print(f"  Normal   (0): {sum(y_train==0):,} "
      f"({sum(y_train==0)/len(y_train)*100:.1f}%)")
print(f"  Abnormal (1): {sum(y_train==1):,} "
      f"({sum(y_train==1)/len(y_train)*100:.1f}%)")

print(f"\nTest Set:")
print(f"  Normal   (0): {sum(y_test==0):,} "
      f"({sum(y_test==0)/len(y_test)*100:.1f}%)")
print(f"  Abnormal (1): {sum(y_test==1):,} "
      f"({sum(y_test==1)/len(y_test)*100:.1f}%)")

### Class imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
sizes  = [sum(y_train==0), sum(y_train==1)]
labels = ['Normal (0)', 'Abnormal (1)']
colors_pie = ['#2ecc71', '#e74c3c']
axes[0].pie(
    sizes, labels=labels, colors=colors_pie,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 13}
)
axes[0].set_title('Class Imbalance in Training Set\n(BEFORE SMOTE)',
                  fontsize=13, fontweight='bold')

# Bar chart
axes[1].bar(labels, sizes, color=colors_pie, edgecolor='black', width=0.5)
axes[1].set_title('Sample Count per Class\n(BEFORE SMOTE)',
                  fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Samples')
for i, v in enumerate(sizes):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontsize=12)

plt.tight_layout()
plt.savefig(FIG_DIR / 'imbalance_before_smote.png', dpi=150)
plt.show()

print("\n⚠️  IMBALANCE PROBLEM DETECTED!")
print(f"   Imbalance ratio: {sizes[0]/sizes[1]:.1f}:1 (Normal:Abnormal)")
print("   This causes models to be biased toward predicting Normal.")
print("   Solution: Apply SMOTE oversampling ✅")

## 5. Balance the training set with SMOTE
SMOTE is fitted on the **training data only**, so no synthetic samples leak into the test set.

In [ ]:
print("=" * 60)
print("HANDLING CLASS IMBALANCE WITH SMOTE")
print("=" * 60)
print("""
What is SMOTE?
  SMOTE = Synthetic Minority Over-sampling TEchnique

How it works:
  - Finds k-nearest neighbors of minority class samples
  - Creates SYNTHETIC new samples between them
  - Does NOT just duplicate — creates realistic new data
  - Result: Balanced dataset without losing majority data

Why NOT just duplicate rows?
  - Duplication = overfitting on repeated samples
  - SMOTE = generates new, diverse minority samples
  - Much better generalization

References:
  - Chawla et al. (2002) — original SMOTE paper
  - Used in Paper 4 (Tan et al.) and Paper 6 (Abedi et al.)
""")

# Apply SMOTE (only on TRAINING data — never on test!)
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_raw, y_train)

print(f"BEFORE SMOTE:")
print(f"  Normal   (0): {sum(y_train==0):,}")
print(f"  Abnormal (1): {sum(y_train==1):,}")
print(f"  Total:        {len(y_train):,}")

print(f"\nAFTER SMOTE:")
print(f"  Normal   (0): {sum(y_train_smote==0):,}")
print(f"  Abnormal (1): {sum(y_train_smote==1):,}")
print(f"  Total:        {len(y_train_smote):,}")
print(f"\n✅ Dataset is now perfectly balanced!")

# Visualize after SMOTE
sizes_after = [sum(y_train_smote==0), sum(y_train_smote==1)]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].pie(
    sizes_after, labels=labels, colors=colors_pie,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 13}
)
axes[0].set_title('Class Distribution\n(AFTER SMOTE)',
                  fontsize=13, fontweight='bold')

axes[1].bar(labels, sizes_after, color=colors_pie, edgecolor='black', width=0.5)
axes[1].set_title('Sample Count per Class\n(AFTER SMOTE)',
                  fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Samples')
for i, v in enumerate(sizes_after):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontsize=12)

plt.tight_layout()
plt.savefig(FIG_DIR / 'balance_after_smote.png', dpi=150)
plt.show()

## 6. Sample ECG signals

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Get one sample from each original class
sample_labels = {
    'Normal (N)': np.where(y_train_raw == 0)[0][0],
    'Supraventricular (S)': np.where(y_train_raw == 1)[0][0],
    'Ventricular (V)': np.where(y_train_raw == 2)[0][0],
    'Fusion (F)': np.where(y_train_raw == 3)[0][0],
    'Unknown (Q)': np.where(y_train_raw == 4)[0][0],
}

colors_ecg = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6', '#3498db']
for idx, (name, sample_idx) in enumerate(sample_labels.items()):
    axes[idx].plot(
        X_train_raw[sample_idx],
        color=colors_ecg[idx], linewidth=1.5
    )
    axes[idx].set_title(f'ECG Signal: {name}', fontsize=11)
    axes[idx].set_xlabel('Time Points (187)')
    axes[idx].set_ylabel('Amplitude')
    axes[idx].grid(True, alpha=0.3)

axes[5].axis('off')
plt.suptitle('Sample ECG Heartbeat Signals by Class',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'ecg_signals.png', dpi=150)
plt.show()

## 7. Random Forest
### Feature extraction
Random Forest works on 13 statistical features per heartbeat rather than the raw sequence.

In [ ]:
print("=" * 50)
print("FEATURE EXTRACTION FOR RANDOM FOREST")
print("=" * 50)
print("Extracting 13 statistical features per heartbeat...")

def extract_statistical_features(X):
    """
    Extract statistical features from each ECG heartbeat.
    Random Forest needs hand-crafted features (not raw sequences).
    """
    features = []
    for row in X:
        f = [
            np.mean(row),             # 1. Mean amplitude
            np.std(row),              # 2. Standard deviation
            np.max(row),              # 3. Maximum value
            np.min(row),              # 4. Minimum value
            np.max(row) - np.min(row),# 5. Range (max-min)
            stats.kurtosis(row),      # 6. Kurtosis (peakedness)
            stats.skew(row),          # 7. Skewness (asymmetry)
            np.median(row),           # 8. Median
            np.percentile(row, 25),   # 9. Q1 (25th percentile)
            np.percentile(row, 75),   # 10. Q3 (75th percentile)
            np.percentile(row,75) -
            np.percentile(row,25),    # 11. IQR (Q3-Q1)
            np.sum(np.abs(np.diff(row))), # 12. Sum of abs differences
            np.sqrt(np.mean(row**2))  # 13. RMS (Root Mean Square)
        ]
        features.append(f)
    return np.array(features)

# Extract features
X_train_rf = extract_statistical_features(X_train_smote)
X_test_rf  = extract_statistical_features(X_test_raw)

# Scale features
scaler = StandardScaler()
X_train_rf_scaled = scaler.fit_transform(X_train_rf)
X_test_rf_scaled  = scaler.transform(X_test_rf)

print(f"✅ Feature extraction complete")
print(f"   Training features shape: {X_train_rf_scaled.shape}")
print(f"   Test features shape:     {X_test_rf_scaled.shape}")
print(f"   Features per heartbeat:  13 statistical measures")

### Training and evaluation

In [ ]:
print("=" * 50)
print("TRAINING RANDOM FOREST CLASSIFIER")
print("=" * 50)

rf_model = RandomForestClassifier(
    n_estimators=100,    # Number of trees
    max_depth=15,        # Max tree depth
    min_samples_split=4, # Min samples to split node
    max_features='sqrt', # Features per split
    class_weight='balanced',
    random_state=42,
    n_jobs=-1            # Use all CPU cores
)

# Train and time it
print("Training Random Forest...")
start_rf = time.time()
rf_model.fit(X_train_rf_scaled, y_train_smote)
rf_train_time = time.time() - start_rf
print(f"✅ Training complete in {rf_train_time:.2f} seconds")

# Inference time
start_inf = time.time()
y_pred_rf = rf_model.predict(X_test_rf_scaled)
rf_inference_time = time.time() - start_inf
y_prob_rf = rf_model.predict_proba(X_test_rf_scaled)[:, 1]

# Metrics
rf_accuracy  = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall    = recall_score(y_test, y_pred_rf)
rf_f1        = f1_score(y_test, y_pred_rf)
rf_auc       = roc_auc_score(y_test, y_prob_rf)

print(f"\n📊 RANDOM FOREST RESULTS:")
print(f"   Accuracy:        {rf_accuracy*100:.2f}%")
print(f"   Precision:       {rf_precision*100:.2f}%")
print(f"   Recall:          {rf_recall*100:.2f}%")
print(f"   F1-Score:        {rf_f1*100:.2f}%")
print(f"   AUC-ROC:         {rf_auc:.4f}")
print(f"   Training time:   {rf_train_time:.2f}s")
print(f"   Inference time:  {rf_inference_time:.4f}s")

# Feature importance
feature_names = [
    'Mean', 'Std Dev', 'Max', 'Min', 'Range',
    'Kurtosis', 'Skewness', 'Median', 'Q1', 'Q3',
    'IQR', 'Sum Abs Diff', 'RMS'
]
importances = rf_model.feature_importances_

fig, ax = plt.subplots(figsize=(10, 5))
sorted_idx = np.argsort(importances)[::-1]
ax.bar(
    range(len(importances)),
    importances[sorted_idx],
    color='#3498db', edgecolor='black'
)
ax.set_xticks(range(len(importances)))
ax.set_xticklabels(
    [feature_names[i] for i in sorted_idx],
    rotation=45, ha='right'
)
ax.set_title('Random Forest Feature Importance',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Importance Score')
plt.tight_layout()
plt.savefig(FIG_DIR / 'rf_feature_importance.png', dpi=150)
plt.show()

## 8. LSTM
### Reshape for sequence input

In [ ]:
print("=" * 50)
print("PREPARING DATA FOR LSTM")
print("=" * 50)

# LSTM needs shape: (samples, timesteps, features)
# Each heartbeat = 187 timesteps, 1 feature (amplitude)
X_train_lstm = X_train_smote.reshape(-1, 187, 1)
X_test_lstm  = X_test_raw.reshape(-1, 187, 1)

print(f"X_train_lstm shape: {X_train_lstm.shape}")
print(f"  → (samples={X_train_lstm.shape[0]}, "
      f"timesteps={X_train_lstm.shape[1]}, "
      f"features={X_train_lstm.shape[2]})")
print(f"X_test_lstm shape:  {X_test_lstm.shape}")

### Build and train

In [ ]:
print("=" * 50)
print("BUILDING LSTM MODEL")
print("=" * 50)

# Build LSTM architecture
lstm_model = Sequential([
    # First LSTM layer
    LSTM(
        64,
        return_sequences=True,
        input_shape=(187, 1)
    ),
    BatchNormalization(),
    Dropout(0.3),

    # Second LSTM layer
    LSTM(
        32,
        return_sequences=False
    ),
    BatchNormalization(),
    Dropout(0.3),

    # Dense layers
    Dense(16, activation='relu'),
    Dropout(0.2),

    # Output layer (binary classification)
    Dense(1, activation='sigmoid')
])

# Compile
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# Train
print("\n🚀 Training LSTM model...")
start_lstm = time.time()

history = lstm_model.fit(
    X_train_lstm, y_train_smote,
    epochs=30,
    batch_size=128,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

lstm_train_time = time.time() - start_lstm
print(f"\n✅ LSTM training complete in {lstm_train_time:.2f} seconds")

### Evaluate

In [ ]:
print("=" * 50)
print("EVALUATING LSTM MODEL")
print("=" * 50)

# Inference
start_inf_lstm = time.time()
y_prob_lstm = lstm_model.predict(X_test_lstm, verbose=0).flatten()
lstm_inference_time = time.time() - start_inf_lstm

# Convert probabilities to binary predictions
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

# Metrics
lstm_accuracy  = accuracy_score(y_test, y_pred_lstm)
lstm_precision = precision_score(y_test, y_pred_lstm)
lstm_recall    = recall_score(y_test, y_pred_lstm)
lstm_f1        = f1_score(y_test, y_pred_lstm)
lstm_auc       = roc_auc_score(y_test, y_prob_lstm)

print(f"\n📊 LSTM RESULTS:")
print(f"   Accuracy:        {lstm_accuracy*100:.2f}%")
print(f"   Precision:       {lstm_precision*100:.2f}%")
print(f"   Recall:          {lstm_recall*100:.2f}%")
print(f"   F1-Score:        {lstm_f1*100:.2f}%")
print(f"   AUC-ROC:         {lstm_auc:.4f}")
print(f"   Training time:   {lstm_train_time:.2f}s")
print(f"   Inference time:  {lstm_inference_time:.4f}s")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['accuracy'],    label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'],label='Validation', linewidth=2)
axes[0].set_title('LSTM Accuracy Over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('LSTM Loss Over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'lstm_training_history.png', dpi=150)
plt.show()

## 9. Model comparison
### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_rf, y_pred_lstm],
    ['Random Forest', 'LSTM'],
    ['Blues', 'Reds']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap=color,
        ax=ax, annot_kws={'size': 14},
        xticklabels=['Normal', 'Abnormal'],
        yticklabels=['Normal', 'Abnormal']
    )
    ax.set_title(f'{title} Confusion Matrix',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)

    # Add TN, FP, FN, TP labels
    tn, fp, fn, tp = cm.ravel()
    ax.text(-0.35, 0.5,
            f'TN={tn:,}\nFP={fp:,}\nFN={fn:,}\nTP={tp:,}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Confusion Matrices: RF vs LSTM',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrices.png', dpi=150)
plt.show()

### ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

# Random Forest ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
ax.plot(fpr_rf, tpr_rf,
        color='#3498db', linewidth=2.5,
        label=f'Random Forest (AUC = {rf_auc:.4f})')

# LSTM ROC
fpr_lstm, tpr_lstm, _ = roc_curve(y_test, y_prob_lstm)
ax.plot(fpr_lstm, tpr_lstm,
        color='#e74c3c', linewidth=2.5,
        label=f'LSTM (AUC = {lstm_auc:.4f})')

# Diagonal (random classifier)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5,
        label='Random Classifier (AUC = 0.50)')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves: Random Forest vs LSTM',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'roc_curves.png', dpi=150)
plt.show()

### Summary table

In [ ]:
print("=" * 65)
print("FINAL MODEL COMPARISON: RANDOM FOREST vs LSTM")
print("=" * 65)

results = {
    'Metric': [
        'Accuracy (%)', 'Precision (%)', 'Recall (%)',
        'F1-Score (%)', 'AUC-ROC',
        'Train Time (s)', 'Inference Time (s)'
    ],
    'Random Forest': [
        f"{rf_accuracy*100:.2f}",
        f"{rf_precision*100:.2f}",
        f"{rf_recall*100:.2f}",
        f"{rf_f1*100:.2f}",
        f"{rf_auc:.4f}",
        f"{rf_train_time:.2f}",
        f"{rf_inference_time:.4f}"
    ],
    'LSTM': [
        f"{lstm_accuracy*100:.2f}",
        f"{lstm_precision*100:.2f}",
        f"{lstm_recall*100:.2f}",
        f"{lstm_f1*100:.2f}",
        f"{lstm_auc:.4f}",
        f"{lstm_train_time:.2f}",
        f"{lstm_inference_time:.4f}"
    ]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print("=" * 65)

# Visual comparison bar chart
metrics   = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
rf_vals   = [rf_accuracy, rf_precision, rf_recall, rf_f1]
lstm_vals = [lstm_accuracy, lstm_precision, lstm_recall, lstm_f1]

x     = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, [v*100 for v in rf_vals],
               width, label='Random Forest',
               color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, [v*100 for v in lstm_vals],
               width, label='LSTM',
               color='#e74c3c', edgecolor='black')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Model Comparison: Random Forest vs LSTM',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim([0, 105])
ax.legend(fontsize=12)
ax.grid(True, axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / 'model_comparison.png', dpi=150)
plt.show()

## 10. Ablation studies
### Effect of SMOTE

In [ ]:
print("=" * 55)
print("ABLATION STUDY 1: Effect of SMOTE on Performance")
print("=" * 55)

# Train Random Forest WITHOUT SMOTE
rf_no_smote = RandomForestClassifier(
    n_estimators=100, max_depth=15,
    random_state=42, n_jobs=-1
)
rf_no_smote.fit(
    extract_statistical_features(X_train_raw),
    y_train
)
y_pred_no_smote = rf_no_smote.predict(
    extract_statistical_features(X_test_raw)
)

print("\nRandom Forest WITHOUT SMOTE:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_no_smote)*100:.2f}%")
print(f"  Recall:    {recall_score(y_test, y_pred_no_smote)*100:.2f}%")
print(f"  F1-Score:  {f1_score(y_test, y_pred_no_smote)*100:.2f}%")

print("\nRandom Forest WITH SMOTE:")
print(f"  Accuracy:  {rf_accuracy*100:.2f}%")
print(f"  Recall:    {rf_recall*100:.2f}%")
print(f"  F1-Score:  {rf_f1*100:.2f}%")

# Visual
categories = ['Accuracy', 'Recall', 'F1-Score']
without_smote = [
    accuracy_score(y_test, y_pred_no_smote)*100,
    recall_score(y_test, y_pred_no_smote)*100,
    f1_score(y_test, y_pred_no_smote)*100
]
with_smote = [rf_accuracy*100, rf_recall*100, rf_f1*100]

x     = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, without_smote, width,
       label='Without SMOTE', color='#95a5a6', edgecolor='black')
ax.bar(x + width/2, with_smote, width,
       label='With SMOTE', color='#2ecc71', edgecolor='black')
ax.set_title('Ablation Study 1: Impact of SMOTE',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Score (%)')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim([60, 105])
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'ablation_smote.png', dpi=150)
plt.show()

### Robustness to Gaussian noise

In [ ]:
print("=" * 55)
print("ABLATION STUDY 2: Robustness to Signal Noise")
print("=" * 55)
print("Adding Gaussian noise at different levels...")

noise_levels = [0.0, 0.01, 0.05, 0.1, 0.2]
rf_noise_f1    = []
lstm_noise_f1  = []

for noise in noise_levels:
    np.random.seed(42)
    # Add noise to test data
    X_test_noisy = X_test_raw + np.random.normal(
        0, noise, X_test_raw.shape
    )

    # RF prediction
    X_test_noisy_rf = extract_statistical_features(X_test_noisy)
    X_test_noisy_rf = scaler.transform(X_test_noisy_rf)
    y_pred_rf_noisy = rf_model.predict(X_test_noisy_rf)
    rf_noise_f1.append(f1_score(y_test, y_pred_rf_noisy)*100)

    # LSTM prediction
    X_test_noisy_lstm = X_test_noisy.reshape(-1, 187, 1)
    y_prob_lstm_noisy = lstm_model.predict(
        X_test_noisy_lstm, verbose=0
    ).flatten()
    y_pred_lstm_noisy = (y_prob_lstm_noisy >= 0.5).astype(int)
    lstm_noise_f1.append(f1_score(y_test, y_pred_lstm_noisy)*100)

    print(f"  Noise={noise:.2f} | RF F1={rf_noise_f1[-1]:.2f}% | "
          f"LSTM F1={lstm_noise_f1[-1]:.2f}%")

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(noise_levels, rf_noise_f1,
        'o-', color='#3498db', linewidth=2.5,
        markersize=8, label='Random Forest')
ax.plot(noise_levels, lstm_noise_f1,
        's-', color='#e74c3c', linewidth=2.5,
        markersize=8, label='LSTM')
ax.set_title('Ablation Study 2: Robustness to Gaussian Noise',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Noise Level (σ)', fontsize=12)
ax.set_ylabel('F1-Score (%)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'ablation_noise.png', dpi=150)
plt.show()

## 11. Classification reports

In [ ]:
print("=" * 50)
print("DETAILED CLASSIFICATION REPORT — RANDOM FOREST")
print("=" * 50)
print(classification_report(
    y_test, y_pred_rf,
    target_names=['Normal', 'Abnormal']
))

print("=" * 50)
print("DETAILED CLASSIFICATION REPORT — LSTM")
print("=" * 50)
print(classification_report(
    y_test, y_pred_lstm,
    target_names=['Normal', 'Abnormal']
))

## 12. Real-time alert simulation

In [ ]:
print("=" * 55)
print("REAL-TIME HEALTH ALERT SIMULATION (LSTM)")
print("=" * 55)
print("Simulating real-time monitoring of incoming heartbeats...")
print()

# Take 10 random samples from test set
np.random.seed(42)
sample_indices = np.random.choice(len(X_test_raw), 10, replace=False)

for i, idx in enumerate(sample_indices):
    heartbeat  = X_test_raw[idx].reshape(1, 187, 1)
    true_label = y_test[idx]
    prob       = lstm_model.predict(heartbeat, verbose=0)[0][0]
    pred_label = int(prob >= 0.5)

    status = "🚨 ALERT: ABNORMAL" if pred_label == 1 else "✅ NORMAL"
    true   = "Abnormal" if true_label == 1 else "Normal"
    correct = "✓" if pred_label == true_label else "✗"

    print(f"  Beat {i+1:2d}: {status} | "
          f"Confidence={prob*100:.1f}% | "
          f"True={true} {correct}")

print()
print("✅ Real-time alert simulation complete")
print("   In production: each heartbeat processed in ~50ms")

## 13. Save models

In [ ]:
import joblib

# Save Random Forest
joblib.dump(rf_model, MODEL_DIR / 'random_forest_ecg.joblib')
joblib.dump(scaler,   MODEL_DIR / 'feature_scaler.joblib')

# Save LSTM
lstm_model.save(MODEL_DIR / 'lstm_ecg_model.keras')

print("✅ Models saved successfully:")
print("   random_forest_ecg.joblib")
print("   feature_scaler.joblib")
print("   lstm_ecg_model.keras")

## 14. Summary

In [ ]:
print()
print("=" * 65)
print("              PROJECT COMPLETE — FINAL SUMMARY")
print("=" * 65)
print(f"""
Dataset:    ECG Heartbeat Categorization (MIT-BIH source)
Task:       Binary Classification (Normal vs Abnormal)
Classes:    0 = Normal, 1 = Abnormal (classes 1-4 merged)
Imbalance:  Handled with SMOTE oversampling

╔══════════════════════╦══════════════╦══════════════╗
║ Metric               ║ Rnd Forest   ║    LSTM      ║
╠══════════════════════╬══════════════╬══════════════╣
║ Accuracy             ║ {rf_accuracy*100:>7.2f}%    ║  {lstm_accuracy*100:>7.2f}%   ║
║ Precision            ║ {rf_precision*100:>7.2f}%    ║  {lstm_precision*100:>7.2f}%   ║
║ Recall               ║ {rf_recall*100:>7.2f}%    ║  {lstm_recall*100:>7.2f}%   ║
║ F1-Score             ║ {rf_f1*100:>7.2f}%    ║  {lstm_f1*100:>7.2f}%   ║
║ AUC-ROC              ║    {rf_auc:.4f}    ║    {lstm_auc:.4f}   ║
║ Training Time        ║ {rf_train_time:>7.2f}s    ║  {lstm_train_time:>7.2f}s   ║
╚══════════════════════╩══════════════╩══════════════╝

Key Findings:
  → Random Forest: higher accuracy and precision (fewer false alarms)
  → LSTM (threshold 0.5): higher recall, but many false positives
  → SMOTE raised Random Forest recall on the abnormal class
  → RF statistical features are sensitive to additive Gaussian noise

Ethical Considerations:
  → Privacy: ECG data requires HIPAA/GDPR compliance
  → Bias: Model performance may vary across demographics
  → Alarm fatigue: Tune the decision threshold to limit false positives
  → Transparency: SHAP could explain RF predictions (future work)
""")
print("=" * 65)